# VAR -  Vector Autoregression Model
http://machinelearningplus.com/time-series/vector-autoregression-examples-python/ <br>
- <b> Autoregressive model </b> --> each variable (Time Series) is modeled as a function of the past values, that is the predictors are nothing but the lags (time delayed value) of the series.
- <b> Bidirectional model </b> --> predictors influence Y AND Y influence predictors = variables influence each other

## Import + setup

In [1]:
%load_ext autoreload
%autoreload 2

In [3]:
from src.config import set_seeds, PLOTS_DIR
import src.pipeline as pipe
import src.models as mod
import src.evaluation as eval
import src.reporting as rep
import src.visualization as visual
import matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import warnings
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import grangercausalitytests
from itertools import permutations
from tqdm import tqdm
from statsmodels.tsa.statespace.varmax import VARMAX
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

In [4]:
set_seeds()

Random seeds set to 42. Deterministic operations enabled.


## Upload

In [5]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


Each variable is modeled as a linear combination of past values of itself and the past values of other variables in the system. Since you have multiple time series that influence each other, it is modeled as a system of equations with one equation per variable (time series).

1. Analyze the time series characteristics
2. Test for causation amongst the time series
3. Test for stationarity
4. Transform the series to make it stationary, if needed
5. Find optimal order (p)
6. Prepare training and test datasets
7. Train the model
8. Roll back the transformations, if any.
9. Evaluate the model using test set
10. Forecast to future

- stationary
- seasonality
- structural breaks
- [spurious correlation](https://statisticsbyjim.com/basics/spurious-correlation/): a spurious correlation occurs when two variables are correlated but don’t have a causal relationship. In other words, it appears like values of one variable cause changes in the other variable, but that’s not actually happening. 

In [ ]:
# def make_stationary(series):
#     def is_stationary(series):
#         result = adfuller(series.dropna())
#         return result[1] < 0.05  # p-value < 0.05 → stazionaria
    
#     if is_stationary(series) : 
#         #print("Serie stazionaria")
#         return series, 0
    
#     current_series = series.copy()
#     for d in range(1, 3):
#         current_series = current_series.diff()
#         if is_stationary(current_series):
#             return current_series, d
    
#     return current_series, 2

- test di stazionarietà = _ADF_ --> rendo tutte le time serie stazionarie + divido tra quelle stazionarie originariamente e quelle che hanno avuto di bisogno di 1 o 2 differenziazioni. 

**Questo ci porterà a 3 gruppi di time series:**
1. stazionarie <br>
2. 1-diff-transformed  -> non sono cointegrate = VAR standard sulle differenze <br>
-> sono cointegrate = VECM (Vector Error Correction Model) <br>
3. 2-diff-transformed <br>

- test di cointegrazione = _Johansen_ --> sulle 1-diff-transformed
    - prerequisiti: no multicollinearità e ritardi p appropriati
    - se r = 0 : nessuna cointegrazione --> VAR standard su tutte e tre i gruppi di prima
    - se r > 0 : sì cointegrazione --> VECM su variabili I(1) [si possono includere I(0), NON possono essere incluse I(2)]

## Loading of Integration Orders

In [7]:
config_path = "../results/integration_orders.xlsx"
if os.path.exists(config_path):
    print(f"Loading configuration from: {config_path}")
    config_df = pd.read_excel(config_path)
    if 'Indicator' in config_df.columns:
        config_df.set_index('Indicator', inplace=True)
    print("Configuration loaded correctly.")
else:
    print(f"ERROR: file {config_path} not found.")

Loading configuration from: ../results/integration_orders.xlsx
Configuration loaded correctly.


In [8]:
def prepare_data_for_var(df, config_df):
    original_I1_series = {}
    original_I2_series = {}
    df_stationary = pd.DataFrame(index=df.index)
    integration_map = {} 

    for col in df.columns:
        try:
            d = int(config_df.loc[col, 'd_full'])
        except IndexError:
            print(f"Warning: Ordine d non trovato per {col}, salto.")
            continue
        integration_map[col] = d
        series = df[col]
        
        if d == 0:
            df_stationary[col] = series
        elif d == 1:
            original_I1_series[col] = series 
            df_stationary[col] = series.diff()
        elif d == 2:
            original_I2_series[col] = series
            df_stationary[col] = series.diff().diff()
        else:
            pass
    df_stationary.dropna(inplace=True)
    return df_stationary, integration_map

## DataFrame containing only stationary series

In [ ]:
df_stationary, transformation_info = prepare_data_for_var(df, config_df)
display(df_stationary.head())

,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1992-01-01,-0.036,-0.169583,-7607.0,-0.438475,-0.544846,-0.237835,0.265037,779.5,-0.258407,-760.0,...,23.883980,2.09,1.25,0.43,671737.0,341.1,-0.121136,7.949339e+08,-0.014675,-0.211437
1993-01-01,-0.060,-0.079171,-22547.0,-0.921662,-0.685508,-1.370476,0.265037,779.5,-0.231206,-680.0,...,69.294369,1.24,-2.98,-5.26,-119439.0,143.5,-0.119202,-9.238378e+09,-0.018946,0.215884
1994-01-01,-0.060,-0.041092,-30262.0,-0.184827,-0.099269,-0.347767,0.265037,779.5,-0.707218,-2080.0,...,-22.354164,0.33,-0.96,-1.73,-584956.0,-175.6,-0.006959,9.517599e+08,0.039501,0.681158
1995-01-01,-0.060,-0.019110,-33807.0,-0.314871,-0.255441,-0.424853,0.265037,779.5,-1.254633,-3690.0,...,-1.555090,3.83,-0.81,-3.17,492552.0,-7.9,-0.013877,2.096193e+09,0.006140,0.038495
1996-01-01,-0.060,0.026185,-28831.0,-0.454062,-0.263117,-0.800961,0.265037,779.5,0.054401,160.0,...,46.442966,1.19,2.99,3.95,1220922.0,280.9,-0.001888,4.061960e+09,-0.031885,-0.869866


In [ ]:
df_metadata = []
stationary_series = {}


for col_name, series in df.items():
    made_stat_serie, d = make_stationary(series)

    stationary_series[col_name] = made_stat_serie
    df_metadata.append({"Indicator": col_name, "d": d})

    if d == 1: original_I1_series[col_name] = series 
    elif d > 1: original_I2_series[col_name] = series

df_metadata = pd.DataFrame(df_metadata).set_index("Indicator")
df_stationary_series = pd.DataFrame(stationary_series)
df_I1_levels = pd.DataFrame(original_I1_series)
df_I2_levels = pd.DataFrame(original_I2_series)

display(df_metadata.head())
display(df_stationary_series.head())
display(df_I1_levels.head())
display(df_I2_levels.head())

if len(df_I1_levels.columns) < 2:
    print("Not enough I(1) variables to execute cointegration test.")
    rango_finale = 0

,d
Indicator,
population_percent,1
population_growth,1
population_abs,2
employment_tot,0
employment_male,0


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,-0.495,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1962-01-01,-0.499,-0.017051,-2810.0,NaN,NaN,NaN,NaN,NaN,-0.105403,-310.0,...,-2.636357,1.68,0.97,0.75,499810.0,43.8,NaN,NaN,2.563660,16.566057
1963-01-01,-0.498,0.039637,8616.0,NaN,NaN,NaN,NaN,NaN,-0.482813,-1420.0,...,-6.187623,-6.64,-0.53,2.76,-1108550.0,-110.1,NaN,NaN,2.651714,14.571604
1964-01-01,-0.497,0.079479,16441.0,NaN,NaN,NaN,NaN,NaN,-0.163204,-480.0,...,0.709619,3.62,3.91,4.12,682860.0,127.8,NaN,NaN,2.792923,15.115329


,population_percent,population_growth,agriland_percent,agriland_abs,arableland_percent,arableland_person,arableland_abs,cerealland_abs,cropland_percent,withdrawals_percent,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_dollars
1960-01-01,40.639,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,70.324028,206830.0,43.731937,0.254510,12862000.0,6387203.0,9.336643,NaN,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN
1962-01-01,39.645,-0.574190,70.218626,206520.0,43.504131,0.251477,12795000.0,6485855.0,9.435245,NaN,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN
1963-01-01,39.147,-0.534554,69.735813,205100.0,43.092720,0.247288,12674000.0,6299467.0,9.319642,NaN,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN
1964-01-01,38.650,-0.455075,69.572609,204620.0,42.834314,0.243791,12598000.0,6244904.0,9.438645,NaN,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN


,population_abs,forestarea_abs,fertilizer_abs,valueadded_percent
1960-01-01,20400656.0,NaN,NaN,NaN
1961-01-01,20287312.0,NaN,67.795055,NaN
1962-01-01,20171158.0,NaN,69.362485,NaN
1963-01-01,20063620.0,NaN,68.764636,NaN
1964-01-01,19972523.0,NaN,74.639625,NaN


In [ ]:
corr_matrix = df_stationary_series.dropna().corr().abs()
upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

CORRELATED_COUPLES = [
    column for column in upper_triangle.columns 
    if any(upper_triangle[column] > 0.99)
]

if CORRELATED_COUPLES:
    print("ATTENZIONE: high multicollinearity found (> 0.99)")
    print("Problematic couples:")

    for col in upper_triangle.columns:
        highly_corr = upper_triangle.index[upper_triangle[col] > 0.99].tolist()
        if highly_corr:
            print(f"- {col} = {highly_corr}")
else:
    print("No perfect multicollinearity (>0.99) found.")

ATTENZIONE: Trovata multicollinearità elevata (> 0.99)
Coppie problematiche:
- population_abs = ['population_growth']
- employment_female = ['employment_tot']
- agriland_abs = ['agriland_percent']
- arableland_person = ['arableland_percent']
- arableland_abs = ['arableland_percent', 'arableland_person']


In [33]:
REDUNDANT = [
    'population_abs',
    'employment_female',
    'employment_male',
    'forestarea_abs',
    'agriland_abs', 
    'arableland_abs', 
    'arableland_person'
]

CATEGORIES = {
    'production' : ['cerealyield_abs',
                    'livestock_production_index',
                    'food_production_index',
                    'crop_production_index',
                    'cereal_production',
                    'valueadded_dollars'],
    'land_use' : ['cropland_percent', 
                    'cerealland_abs',
                    'arableland_percent',
                    'agriland_percent'],
    'prerequisites' : ['withdrawals_percent',
                    'fertilizer_percent']
}

In [ ]:
df_I1_johansen = df_I1_levels.drop(columns=REDUNDANT, errors='ignore')
cointegration_results = {}

for category_name, var_list in CATEGORIES.items():
    print(f"\n=======================================================")
    print(f"CATEGORY: {category_name.upper()}")
    print(f"=======================================================")
    
    # Seleziona le vars I(1) pulite per questo gruppo
    # (intersezione tra le vars I(1) e quelle della categoria)
    vars_to_test = [col for col in var_list if col in df_I1_johansen.columns]
    
    if len(vars_to_test) < 2:
        print(f"Test saltato: meno di 2 vars I(1) in questo gruppo ({vars_to_test}).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Saltato - Poche vars I(1)', 'vars': vars_to_test}
        continue
        
    print(f"vars I(1) in test: {vars_to_test}")
    subgroup_df = df_I1_johansen[vars_to_test].dropna()
    
    if subgroup_df.shape[0] < 20:
        print(f"Test saltato: dati insufficienti dopo dropna ({subgroup_df.shape[0]} righe).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Saltato - Dati insuff.', 'vars': vars_to_test}
        continue

    try:
        var_model_sub = VAR(subgroup_df)
        
        safe_maxlags = min(4, subgroup_df.shape[0] - 1)
        safe_maxlags = max(1, safe_maxlags)
        
        selected_result = var_model_sub.select_order(maxlags=safe_maxlags)
        p_lags = selected_result.selected_orders['aic']
        if p_lags == 0:
            p_lags = 1
        k_ar_diff = p_lags - 1
        print(f"Ritardo ottimale (p): {p_lags} | k_ar_diff (p-1): {k_ar_diff}")
    except Exception as e:
        print(f"Errore selezione ritardi, uso k_ar_diff=1 (p=2). Errore: {e}")
        k_ar_diff = 1 # Fallback
        
    try:
        johansen_result = coint_johansen(
            subgroup_df,
            det_order=1,
            k_ar_diff=k_ar_diff
        )

        trace_rank = 0
        significance_level = 1 # 5%
        
        print("H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione")
        
        for i in range(len(subgroup_df.columns)):
            stat = johansen_result.lr1[i]
            crit = johansen_result.cvt[i, significance_level]

            if np.isnan(crit):
                print(f"H0: r <= {i:<2} | {stat:<13.2f} | {crit:<17.2f} | ERRORE (nan)")
                break
            
            decision = "Rifiuta H0"
            if stat > crit:
                trace_rank = i + 1
            else:
                decision = "Non Rifiuta H0"
            
            print(f"H0: r <= {i:<2} | {stat:<13.2f} | {crit:<17.2f} | {decision}")
            
            if decision == "Non Rifiuta H0":
                break
                
        final_rank = trace_rank
        print(f"\n--> rank di Cointegrazione (r) per '{category_name}': {final_rank}")
        
        if final_rank > 0:
            print("--> CONCLUSIONE: Le serie in questo gruppo sono COINTEGRATE (usare VECM).")
            cointegration_results[category_name] = {'rank': final_rank, 'status': 'Cointegrato', 'vars': vars_to_test}
        else:
            print("--> CONCLUSIONE: Le serie non sono cointegrate (usare VAR in differenze).")
            cointegration_results[category_name] = {'rank': 0, 'status': 'Non Cointegrato', 'vars': vars_to_test}

    except np.linalg.LinAlgError:
        print("ERRORE: LinAlgError (Matrice non definita positiva).")
        print("--> Causa probabile: multicollinearità residua o dati insufficienti.")
        print("--> CONCLUSIONE: Tratto come Non Cointegrato (r=0).")
        cointegration_results[category_name] = {'rank': 0, 'status': 'Errore (LinAlg)', 'vars': vars_to_test}

print("\n=======================================================")
print("RIEPILOGO TEST DI COINTEGRAZIONE")
print("=======================================================")
print(f"{'Categoria':<15} | {'rank (r)':<8} | {'Status':<20} | vars Testate")
print("-" * 80)

for cat, res in cointegration_results.items():
    vars_str = ', '.join(res['vars']) if res['vars'] else 'N/A'
    print(f"{cat:<15} | {res['rank']:<8} | {res['status']:<20} | {vars_str}")


CATEGORIA: PRODUCTION
vars I(1) in test: ['cerealyield_abs', 'livestock_production_index', 'food_production_index', 'crop_production_index', 'cereal_production', 'valueadded_dollars']
Errore selezione ritardi, uso k_ar_diff=1 (p=2). Errore: maxlags is too large for the number of observations and the number of equations. The largest model cannot be estimated.
H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione
H0: r <= 0  | 120.92        | 107.34            | Rifiuta H0
H0: r <= 1  | 77.40         | 79.34             | Non Rifiuta H0

--> rank di Cointegrazione (r) per 'production': 1
--> CONCLUSIONE: Le serie in questo gruppo sono COINTEGRATE (usare VECM).

CATEGORIA: LAND_USE
vars I(1) in test: ['cropland_percent', 'cerealland_abs', 'arableland_percent', 'agriland_percent']
Ritardo ottimale (p): 1 | k_ar_diff (p-1): 0
H0: r <= k  | Stat. Traccia | Val. Critico (5%) | Decisione
H0: r <= 0  | 49.86         | 55.25             | Non Rifiuta H0

--> rank di Cointegrazione (r) per

- r = 0 --> non hanno una relazione di equilibrio stabile nel lungo periodo. Anche se vagano (essendo I(1)), non sono "legate" l'una all'altra. Posso usare VAR.
- r > 0 --> hanno una relazione di equilibrio stabile nel lungo periodo. 
    - Questo gruppo di 5 variabili è "legato" da 3 relazioni di lungo periodo (guinzagli). Anche se le singole serie vagano, queste 3 relazioni le costringono a muoversi insieme nel tempo. Queste variabili devono essere modellate usando un VECM (Vector Error Correction Model), che modella sia le dinamiche di breve periodo (le differenze) sia il ritorno all'equilibrio di lungo periodo (l'Error Correction Term).

In [35]:
# escludiamo vars cointegrate che non possono essere studiate da VAR ma necessitano di VECM
excluded_from_var = []
for cat, res in cointegration_results.items():
    if res['rank'] > 0:
        excluded_from_var.extend(res['vars'])

# escludiamo anche redundant che non mi interessano
for item in REDUNDANT:
    if item not in excluded_from_var:
        excluded_from_var.append(item)
# print(f"vars cointegrate (r > 0) o ridondanti da escludere dal VAR: {excluded_from_var}")

# prendo tutte le serie rese stazionarie 
# + tolgo le vars cointegrate e quelle ridondanti 
# + tolgo nan che non possono esserci per VAR
df_var_input = df_stationary_series.copy()
df_var_input = df_var_input.drop(columns=excluded_from_var, errors='ignore')
df_var_input = df_var_input.dropna()

print(df_var_input.shape)
display(df_var_input.head())

(31, 12)


,population_percent,population_growth,employment_tot,forestarea_percent,agriland_percent,arableland_percent,cerealland_abs,cropland_percent,fertilizer_abs,valueadded_percent,exports_percent,imports_percent
1992-01-01,-0.036,-0.169583,8.004366,26.335895,-0.258407,-0.510013,-176287.0,-0.241406,-11.035499,-0.177889,0.640307,4.709148
1993-01-01,-0.060,-0.079171,7.082705,26.600932,-0.231206,-0.680018,-149640.0,-0.166604,6.587672,0.001933,0.621360,4.925032
1994-01-01,-0.060,-0.041092,6.897878,26.865969,-0.707218,-0.751420,27970.0,-0.054401,1.482552,0.112244,0.660861,5.606190
1995-01-01,-0.060,-0.019110,6.583007,27.131005,-1.254633,-0.156404,112500.0,-0.574615,-11.717296,-0.006918,0.667001,5.644685
1996-01-01,-0.060,0.026185,6.128945,27.396042,0.054401,0.166604,7398.0,0.098603,11.080728,0.011989,0.635116,4.774820


In [36]:
max_lag_granger = 3

def test_granger_pair(data, causing_var, caused_var, maxlag):
    """
    Esegue il test di Granger per una singola coppia e restituisce il p-value minimo.
    H0: 'causing_var' NON causa (Granger) 'caused_var'
    """
    test_data = data[[caused_var, causing_var]]

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try:
            results = grangercausalitytests(test_data, maxlag=maxlag, verbose=False)
        except Exception as e:
            print(f"Errore testando {causing_var} -> {caused_var}: {e}")
            return np.nan

    # Estraiamo il p-value minimo tra tutti i ritardi testati
    # Ci interessa il test 'ssr_ftest' (colonna 1)
    min_p_value = 1.0
    for lag in range(1, maxlag + 1):
        p_value = results[lag][0]['ssr_ftest'][1]
        if p_value < min_p_value:
            min_p_value = p_value
    return min_p_value

variables = df_var_input.columns
granger_results = []

# 'permutations' testa sia (A, B) che (B, A)
for pair in permutations(variables, 2):
    causing_var = pair[0]
    caused_var = pair[1]

    p_value = test_granger_pair(df_var_input, causing_var, caused_var, max_lag_granger)
    
    granger_results.append({
        "Variabile Causa (X)": causing_var,
        "Variabile Effetto (Y)": caused_var,
        "Min P-Value": p_value
    })

df_granger_summary = pd.DataFrame(granger_results)
df_granger_summary = df_granger_summary.sort_values(by="Min P-Value")
display(df_granger_summary.head(10))

,Variabile Causa (X),Variabile Effetto (Y),Min P-Value
32,employment_tot,imports_percent,0.002420
27,employment_tot,cerealland_abs,0.003566
66,cerealland_abs,population_percent,0.014914
73,cerealland_abs,fertilizer_abs,0.019967
94,fertilizer_abs,cerealland_abs,0.023303
60,arableland_percent,cerealland_abs,0.026710
49,agriland_percent,cerealland_abs,0.026898
20,population_growth,exports_percent,0.030284
110,exports_percent,population_percent,0.032297
33,forestarea_percent,population_percent,0.034874


*Domande che mi devo fare per coppie:*
1. Do they make sense as causal relationships? <br> 
2. Do they fit established theory? <br>
3. Can you find a mechanism for causation? <br>
4. Is there a direct link, or are mediator variables involved? <br>

*Scrematura logica dei risultati della Granger causality:*
- **Employment in agriculture - Agricultural raw materials imports** = meno persone lavorano nell'agricoltura più bisogna importare?
- **Employment in agriculture - Land under cereal production** = meno persone lavorano nell'agricoltura più diminuisce la superficie dei campi di cereali
- **Land under cereal production - Fertilizer consumption** = più aumenta la terra arata a cereali più aumenta il consumo di fertilizzanti (questo ci sta!!)
- Arable land - Land under cereal production = meno terra arabile, meno terra arata a cereali (boh sembra stupido)

In [37]:
def optimize_VARMAX(endog: pd.DataFrame, max_p_to_try: int) -> int:
    results = []
    for i in tqdm(range(1, max_p_to_try + 1), desc="Ottimizzazione AIC"):
        try:
            model = VARMAX(endog, order=(i, 0)).fit(disp=False)
            results.append({'p': i, 'aic': model.aic})
        except Exception:
            break
            
    if not results:
        return 1 # Fallback

    result_df = pd.DataFrame(results)
    best_p = result_df.loc[result_df['aic'].idxmin()]['p']
    return int(best_p)

Per ogni coppia che ho individuato:
1. trovo la p ottimale
2. stimo modello VARMAX(p,0)
3. faccio residual analysis
    - QQ plot: i residui sono "normali"? --> i punti devono seguire la linea rossa
    - ACF plot: i residui sono "non correlati"? --> le barre devono stare nell'area blu
    - Ljung-box test: versione numerica dell'ACF plot --> p-value deve essere > 0.05

In [38]:
CAUSEEFFECT_COUPLES = [
    ['employment_tot', 'imports_percent'],
    ['employment_tot', 'cerealland_abs'],
    ['cerealland_abs', 'fertilizer_abs']
]

_selected = []
for pair in CAUSEEFFECT_COUPLES:
    for v in pair:
        if v not in _selected:
            _selected.append(v)

# Filtra solo quelle presenti in df_var_input
found = [v for v in _selected if v in df_var_input.columns]
df_causeeffect = df_var_input[found].copy()

In [39]:
original_index = df_causeeffect.index
original_columns = df_causeeffect.columns

scaler = StandardScaler()
df_causeeffect_scaled_array = scaler.fit_transform(df_causeeffect)
df_causeeffect_scaled = pd.DataFrame(df_causeeffect_scaled_array, 
                                index=original_index, 
                                columns=original_columns)

display(df_causeeffect_scaled.head())

,employment_tot,imports_percent,cerealland_abs,fertilizer_abs
1992-01-01,2.761439,1.387728,-0.954729,-0.592224
1993-01-01,1.995819,1.568295,-0.761135,0.371411
1994-01-01,1.842284,2.138021,0.529223,0.092263
1995-01-01,1.580723,2.170219,1.143343,-0.629505
1996-01-01,1.203535,1.442657,0.379765,0.617091


In [40]:
TRAIN_LEN = int(len(df_causeeffect_scaled) * 0.8)
HORIZON = len(df_causeeffect_scaled) - TRAIN_LEN
WINDOW = 1

_pairs = []
for item in CAUSEEFFECT_COUPLES:
    _pairs.append((item[0], item[1]))

summary_results = {}
for causing, caused in _pairs:
    print(f"ANALISI COPPIA: '{causing}' -> '{caused}'")
    df_pair = df_causeeffect_scaled[[causing, caused]].dropna()[:TRAIN_LEN]
    n_obs = df_pair.shape[0]
    safe_max_p = max(1, TRAIN_LEN // 3)
    if safe_maxlags < 1:
        print(f"Pochi dati ({n_obs} osservazioni).")
        continue
    
    p_selected = optimize_VARMAX(df_pair, safe_maxlags)
    print(f"Ritardo ottimale (AIC) scelto (p): {p_selected}")
    fitted_model = VARMAX(df_pair, order=(p_selected, 0)).fit(disp=False)
    # print(fitted_model.summary())
    
    resid = fitted_model.resid
    
    ljung_pvals = {}
    for col in resid.columns:
        lb_df = acorr_ljungbox(resid[col].dropna(), lags=[10], return_df=True)
        pval = lb_df['lb_pvalue'].iloc[0]
        ljung_pvals[col] = pval
        print(f"Ljung-Box p-value ({col}): {pval:.4f}")
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(f"Analisi Residui per VAR({p_selected})", fontsize=20, fontweight='bold')

    qqplot(resid[causing].dropna(), line='s', ax=axes[0, 0])
    axes[0, 0].set_title(f"QQ-Plot Residui: {causing}")
    qqplot(resid[caused].dropna(), line='s', ax=axes[0, 1])
    axes[0, 1].set_title(f"QQ-Plot Residui: {caused}")

    plot_acf(resid[causing].dropna(), ax=axes[1, 0], lags=20)
    axes[1, 0].set_title(f"ACF Residui: {causing}")
    plot_acf(resid[caused].dropna(), ax=axes[1, 1], lags=20)
    axes[1, 1].set_title(f"ACF Residui: {caused}")
    
    plt.tight_layout()
    plt.show()
    
    summary_results[(causing, caused)] = {
        "p_selected": p_selected,
        "n_obs": n_obs,
        "ljungbox_pvals": ljung_pvals,
        "fitted_model": fitted_model
    }
    print('\n')

ANALISI COPPIA: 'employment_tot' -> 'imports_percent'


Ottimizzazione AIC: 100%|██████████| 4/4 [00:17<00:00,  4.43s/it]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (employment_tot): 0.9894
Ljung-Box p-value (imports_percent): 0.7957


ANALISI COPPIA: 'employment_tot' -> 'cerealland_abs'


Ottimizzazione AIC: 100%|██████████| 4/4 [00:17<00:00,  4.34s/it]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (employment_tot): 0.9775
Ljung-Box p-value (cerealland_abs): 0.7188


ANALISI COPPIA: 'cerealland_abs' -> 'fertilizer_abs'


Ottimizzazione AIC: 100%|██████████| 4/4 [00:05<00:00,  1.45s/it]


Ritardo ottimale (AIC) scelto (p): 1
Ljung-Box p-value (cerealland_abs): 0.4923
Ljung-Box p-value (fertilizer_abs): 0.1024




In [41]:
def rolling_forecast(df, causing, caused, p, train_len, horizon, window, method):
    total_len = train_len + horizon
    
    if method == 'last':
        causing_pred_last = []
        caused_pred_last = []
        for i in tqdm(range(train_len, total_len, window)):
            causing_last = df[:i].iloc[-1][causing]
            caused_last = df[:i].iloc[-1][caused]
            causing_pred_last.extend(causing_last for _ in range(window))
            caused_pred_last.extend(caused_last for _ in range(window))
        return causing_pred_last, caused_pred_last
    
    elif method == 'VAR':
        causing_pred_VAR = []
        caused_pred_VAR = []
        for i in tqdm(range(train_len, total_len, window)):
            try:
                scaled_test = df[[causing, caused]][:i]
                model = VARMAX(scaled_test, order=(p, 0))
                res = model.fit(disp=False)

                predictions_scaled = res.forecast(steps=window)
                preds_arr = np.asarray(predictions_scaled)

                if preds_arr.ndim == 1:
                    preds_arr = preds_arr.reshape(1, -1)

                oos_pred_causing = preds_arr[:, 0]
                oos_pred_caused = preds_arr[:, 1]

                causing_pred_VAR.extend(oos_pred_causing.tolist())
                caused_pred_VAR.extend(oos_pred_caused.tolist())

            except (np.linalg.LinAlgError, ValueError) as e:
                print(f"Errore stima a i={i}: {e}. Inserimento NaN.")
                causing_pred_VAR.extend([np.nan] * window)
                caused_pred_VAR.extend([np.nan] * window)
        return causing_pred_VAR, caused_pred_VAR

In [42]:
TRAIN_LEN = int(len(df_causeeffect_scaled) * 0.8)
HORIZON = len(df_causeeffect_scaled) - TRAIN_LEN
WINDOW = 1

forecast_results_scaled_diff = {}
scaled_diff_test = df_causeeffect_scaled.iloc[TRAIN_LEN : TRAIN_LEN + HORIZON]

for pair_key, results in summary_results.items():
    causing, caused = pair_key
    pair_vars = [causing, caused]
    optimal_p = results['p_selected']
    print(pair_key)
    causing_pred_last, caused_pred_last = rolling_forecast(df_causeeffect_scaled, causing, caused, optimal_p, TRAIN_LEN, HORIZON, WINDOW, 'last')
    causing_pred_VAR, caused_pred_VAR = rolling_forecast(df_causeeffect_scaled, causing, caused, optimal_p, TRAIN_LEN, HORIZON, WINDOW, 'VAR')
    
    df_test_pair_scaled_diff = pd.DataFrame({
        f"{causing}_actual": scaled_diff_test[causing],
        f"{caused}_actual": scaled_diff_test[caused],
        f"{causing}_pred_last": causing_pred_last,
        f"{caused}_pred_last": caused_pred_last,
        f"{causing}_pred_VAR": causing_pred_VAR,
        f"{caused}_pred_VAR": caused_pred_VAR,
    }, index=scaled_diff_test.index)
    
    forecast_results_scaled_diff[pair_key] = df_test_pair_scaled_diff

display(forecast_results_scaled_diff)

('employment_tot', 'imports_percent')


100%|██████████| 7/7 [00:10<00:00,  1.44s/it]


('employment_tot', 'cerealland_abs')


100%|██████████| 7/7 [00:10<00:00,  1.44s/it]


('cerealland_abs', 'fertilizer_abs')


100%|██████████| 7/7 [00:02<00:00,  3.10it/s]


{('employment_tot',
  'imports_percent'):             employment_tot_actual  imports_percent_actual  \
 2016-01-01              -0.660958               -0.684234   
 2017-01-01              -0.745380               -0.926109   
 2018-01-01              -0.766928               -0.895883   
 2019-01-01              -0.656454               -0.993337   
 2020-01-01              -0.579957               -1.106471   
 2021-01-01              -0.523293               -1.071983   
 2022-01-01              -0.741328               -1.055820   
 
             employment_tot_pred_last  imports_percent_pred_last  \
 2016-01-01                 -0.771498                  -0.605878   
 2017-01-01                 -0.660958                  -0.684234   
 2018-01-01                 -0.745380                  -0.926109   
 2019-01-01                 -0.766928                  -0.895883   
 2020-01-01                 -0.656454                  -0.993337   
 2021-01-01                 -0.579957                

In [43]:
def inverse_scale_dataframe(df_pair_diff, scaler, original_scaler_columns, pair_key):
    df_pair_unscaled_diff = pd.DataFrame(index=df_pair_diff.index)
    causing, caused = pair_key

    try:
        idx_causing = list(original_scaler_columns).index(causing)
        idx_caused = list(original_scaler_columns).index(caused)
    except ValueError as e:
        print(f"Errore: una variabile della coppia {pair_key} non era nello scaler originale. {e}")
        return None

    n_samples = len(df_pair_diff)
    n_features = len(original_scaler_columns)

    for col_suffix in ["_actual", "_pred_last", "_pred_VAR"]:
        dummy_data = np.zeros((n_samples, n_features))
        col_causing = f"{causing}{col_suffix}"
        col_caused = f"{caused}{col_suffix}"
        
        dummy_data[:, idx_causing] = df_pair_diff[col_causing].values
        dummy_data[:, idx_caused] = df_pair_diff[col_caused].values
        
        inverted_array = scaler.inverse_transform(dummy_data)
        
        df_pair_unscaled_diff[col_causing] = inverted_array[:, idx_causing]
        df_pair_unscaled_diff[col_caused] = inverted_array[:, idx_caused]
        
    return df_pair_unscaled_diff

In [44]:
def inverse_difference_dataframe(df_pair_unscaled_diff, wide_df, df_metadata, pair_key, train_len):
    df_pair_level = df_pair_unscaled_diff.copy()
    causing, caused = pair_key

    for var_name in [causing, caused]:
        d = df_metadata.loc[var_name]['d']
        # print(var_name, d)       # GIUSTO
        for col_suffix in ["_actual", "_pred_last", "_pred_VAR"]:
            col_name = f"{var_name}{col_suffix}"
            preds_unscaled_diff = df_pair_unscaled_diff[col_name].values
            
            preds_level = []

            if d == 0:
                preds_level = preds_unscaled_diff
                
            elif d == 1:
                last_known_level = wide_df[var_name].iloc[train_len - 1]
                preds_level = last_known_level + np.cumsum(preds_unscaled_diff)
                
            elif d == 2:
                last_known_level = wide_df[var_name].iloc[train_len - 1]
                last_known_diff1 = wide_df[var_name].diff().iloc[train_len - 1]
                
                preds_diff1_level = last_known_diff1 + np.cumsum(preds_unscaled_diff)
                preds_level = last_known_level + np.cumsum(preds_diff1_level)
            df_pair_level[col_name] = preds_level
            
    return df_pair_level

In [45]:
original_scaler_columns = list(df_causeeffect.columns)
forecast_results_level = {}

for pair_key, df_scaled_diff in forecast_results_scaled_diff.items():
    print(f"Inversione per la coppia: {pair_key}")

    df_unscaled_diff = inverse_scale_dataframe(
        df_scaled_diff,
        scaler,
        original_scaler_columns,
        pair_key
    )
    
    df_level = inverse_difference_dataframe(
        df_unscaled_diff,
        df,
        df_metadata,
        pair_key,
        TRAIN_LEN
    )
    forecast_results_level[(causing, caused)] = df_level
    display(df_level.head())

print("\n--- Confronto: Risultati 'scaled_diff' (Scalati e Differenziati) ---")
display(forecast_results_scaled_diff)

Inversione per la coppia: ('employment_tot', 'imports_percent')


,employment_tot_actual,imports_percent_actual,employment_tot_pred_last,imports_percent_pred_last,employment_tot_pred_VAR,imports_percent_pred_VAR
2016-01-01,3.884447,2.231932,3.751378,2.325614,3.719141,2.205362
2017-01-01,3.782819,1.942749,3.884447,2.231932,3.890105,2.141149
2018-01-01,3.756879,1.978887,3.782819,1.942749,3.823840,1.860768
2019-01-01,3.889869,1.862372,3.756879,1.978887,3.785201,1.896459
2020-01-01,3.981957,1.727109,3.889869,1.862372,3.955062,1.804541


Inversione per la coppia: ('employment_tot', 'cerealland_abs')


,employment_tot_actual,cerealland_abs_actual,employment_tot_pred_last,cerealland_abs_pred_last,employment_tot_pred_VAR,cerealland_abs_pred_VAR
2016-01-01,3.884447,5105542.0,3.751378,5010033.0,3.789894,5.021201e+06
2017-01-01,3.782819,4992136.0,3.884447,4988512.0,3.849432,4.950400e+06
2018-01-01,3.756879,4944257.0,3.782819,4875106.0,3.820478,4.859888e+06
2019-01-01,3.889869,4918077.0,3.756879,4827227.0,3.751337,4.775587e+06
2020-01-01,3.981957,4863287.0,3.889869,4801047.0,3.853018,4.714067e+06


Inversione per la coppia: ('cerealland_abs', 'fertilizer_abs')


,cerealland_abs_actual,fertilizer_abs_actual,cerealland_abs_pred_last,fertilizer_abs_pred_last,cerealland_abs_pred_VAR,fertilizer_abs_pred_VAR
2016-01-01,5105542.0,230.305564,5010033.0,250.797063,5.126327e+06,239.890155
2017-01-01,4992136.0,236.006551,4988512.0,265.239532,5.046561e+06,256.420518
2018-01-01,4944257.0,242.174705,4875106.0,281.242848,5.014013e+06,275.413581
2019-01-01,4918077.0,247.109291,4827227.0,297.713333,4.970478e+06,293.791034
2020-01-01,4863287.0,255.153966,4801047.0,312.950248,4.919660e+06,311.128466



--- Confronto: Risultati 'scaled_diff' (Scalati e Differenziati) ---


{('employment_tot',
  'imports_percent'):             employment_tot_actual  imports_percent_actual  \
 2016-01-01              -0.660958               -0.684234   
 2017-01-01              -0.745380               -0.926109   
 2018-01-01              -0.766928               -0.895883   
 2019-01-01              -0.656454               -0.993337   
 2020-01-01              -0.579957               -1.106471   
 2021-01-01              -0.523293               -1.071983   
 2022-01-01              -0.741328               -1.055820   
 
             employment_tot_pred_last  imports_percent_pred_last  \
 2016-01-01                 -0.771498                  -0.605878   
 2017-01-01                 -0.660958                  -0.684234   
 2018-01-01                 -0.745380                  -0.926109   
 2019-01-01                 -0.766928                  -0.895883   
 2020-01-01                 -0.656454                  -0.993337   
 2021-01-01                 -0.579957                

In [46]:
import matplotlib.dates as mdates

def plot_forecast_comparison(train_data_full, test_data_pair, 
                            causing, caused,
                            optimal_p, TRAIN_LEN):
    train_end_index = train_data_full.index[TRAIN_LEN] 
    
    actual_causing_col = f"{causing}_actual"
    last_causing_col = f"{causing}_pred_last"
    var_causing_col = f"{causing}_pred_VAR"
    
    actual_caused_col = f"{caused}_actual"
    last_caused_col = f"{caused}_pred_last"
    var_caused_col = f"{caused}_pred_VAR"

    train_causing_series = train_data_full[causing]
    test_causing_actual = test_data_pair[actual_causing_col]
    test_causing_last = test_data_pair[last_causing_col]
    test_causing_var = test_data_pair[var_causing_col]

    train_caused_series = train_data_full.loc[:train_end_index, caused]
    test_caused_actual = test_data_pair[actual_caused_col]
    test_caused_last = test_data_pair[last_caused_col]
    test_caused_var = test_data_pair[var_caused_col]
    
    fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(16, 10))
    fig.suptitle(f"Forecast vs. Actual [{causing} - {caused}] --> VAR(p={optimal_p})", fontsize=18, fontweight="bold")
    
    for axes in ax.flatten():
        axes.xaxis_date()
        axes.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    
    ax[0, 0].plot(train_causing_series.index, train_causing_series.values, label='Train (Actual)', color=colors[0])
    ax[0, 0].plot(test_causing_actual.index, test_causing_actual.values, color=colors[0], linestyle='-', label='Test (Actual)', marker='o', markersize=4)
    ax[0, 0].plot(test_causing_last.index, test_causing_last.values, color=colors[1], linestyle='-.', label='Pred (Last Value)', marker='^', markersize=4)
    ax[0, 0].plot(test_causing_var.index, test_causing_var.values, color=colors[2], linestyle='--', label='Pred (VAR)', marker='s', markersize=4)
    ax[0, 0].set_title(f"Full Series: {causing}", fontsize=12)
    ax[0, 0].set_ylabel('Value (Original Scale)')
    ax[0, 0].grid(True, alpha=0.3)
    ax[0, 0].legend(loc='best')
    start_year = test_causing_actual.index.min()
    end_year = test_causing_actual.index.max()
    ax[0, 0].axvspan(start_year, end_year, color='#808080', alpha=0.15)
    ax[0, 0].set_xlabel('Year')

    ax[0, 1].plot(test_causing_actual.index, test_causing_actual.values, color=colors[0], linestyle='-', label='Test (Actual)', marker='o')
    ax[0, 1].plot(test_causing_last.index, test_causing_last.values, color=colors[1], linestyle='-.', label='Pred (Last Value)', marker='^')
    ax[0, 1].plot(test_causing_var.index, test_causing_var.values, color=colors[2], linestyle='--', label='Pred (VAR)', marker='s')
    ax[0, 1].set_title(f"Zoom (Test Set): {causing}", fontsize=12)
    ax[0, 1].grid(True, alpha=0.3)
    ax[0, 1].legend(loc='best')

    ax[1, 0].plot(train_caused_series.index, train_caused_series.values, label='Train (Actual)', color=colors[0])
    ax[1, 0].plot(test_caused_actual.index, test_caused_actual.values, color=colors[0], linestyle='-', label='Test (Actual)', marker='o', markersize=4)
    ax[1, 0].plot(test_caused_last.index, test_caused_last.values, color=colors[1], linestyle='-.', label='Pred (Last Value)', marker='^', markersize=4)
    ax[1, 0].plot(test_caused_var.index, test_caused_var.values, color=colors[2], linestyle='--', label='Pred (VAR)', marker='s', markersize=4)
    ax[1, 0].set_title(f"Full Series: {caused}", fontsize=12)
    ax[1, 0].set_xlabel('Year')
    ax[1, 0].set_ylabel('Value (Original Scale)')
    ax[1, 0].grid(True, alpha=0.3)
    ax[1, 0].legend(loc='best')
    ax[1, 0].axvspan(start_year, end_year, color='#808080', alpha=0.15)
    
    ax[1, 1].plot(test_caused_actual.index, test_caused_actual.values, color=colors[0], linestyle='-', label='Test (Actual)', marker='o')
    ax[1, 1].plot(test_caused_last.index, test_caused_last.values, color=colors[1], linestyle='-.', label='Pred (Last Value)', marker='^')
    ax[1, 1].plot(test_caused_var.index, test_caused_var.values, color=colors[2], linestyle='--', label='Pred (VAR)', marker='s')
    ax[1, 1].set_title(f"Zoom (Test Set): {caused}", fontsize=12)
    ax[1, 1].grid(True, alpha=0.3)
    ax[1, 1].legend(loc='best')

    plt.tight_layout(rect=[0, 0.03, 1, 1])
    filename = f"forecast_comparison_{causing}_{caused}.png"
    full_path = os.path.join(PLOTS_DIR, filename)
    try:
        plt.savefig(full_path, dpi=300, bbox_inches='tight')
        print(f"Plot saved: {full_path}")
    except Exception as e:
        print(f"Error saving plot: {e}")
    plt.show()

In [47]:
for pair_key, results in summary_results.items():
    causing, caused = pair_key
    optimal_p = results['p_selected']
    
    if pair_key not in forecast_results_scaled_diff:
        print(f"ATTENZIONE: Chiave {pair_key} non trovata in forecast_results_scaled_diff. Salto il plot.")
        continue
    test_df = forecast_results_scaled_diff[pair_key]
    
    print(f"\n{pair_key}")
    plot_forecast_comparison(
        train_data_full=df.dropna(),
        test_data_pair=test_df,
        causing =causing,
        caused = caused,
        optimal_p=optimal_p,
        TRAIN_LEN=TRAIN_LEN 
    )


('employment_tot', 'imports_percent')
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\forecast_comparison_employment_tot_imports_percent.png

('employment_tot', 'cerealland_abs')
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\forecast_comparison_employment_tot_cerealland_abs.png

('cerealland_abs', 'fertilizer_abs')
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\forecast_comparison_cerealland_abs_fertilizer_abs.png
